In [2]:
!apt install ffmpeg -y -q
!pip install gradio opencv-python-headless matplotlib numpy pandas scikit-image -q

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [3]:
%%writefile gradio_app.py

import os
import cv2
import shutil
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr

# =====================================================
# SETUP & UTILS
# =====================================================
UPLOAD_DIR = "uploads"
FRAME_DIR = "frames"
os.makedirs(UPLOAD_DIR, exist_ok=True)
os.makedirs(FRAME_DIR, exist_ok=True)

cache_metrik = {}
cache_motion = {}
total_frames_global = 0

def get_video_info(mp4_path, raw_path):
    cap = cv2.VideoCapture(mp4_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_f = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_f / fps if fps > 0 else 0
    cap.release()

    size_mp4 = os.path.getsize(mp4_path) / (1024 * 1024)
    size_raw = os.path.getsize(raw_path) / (1024 * 1024) if os.path.exists(raw_path) else 0
    ratio = size_raw / size_mp4 if size_mp4 > 0 else 0

    info_df = pd.DataFrame([
        ["FPS", round(fps, 2)],
        ["Resolusi", f"{w} x {h}"],
        ["Durasi (s)", round(duration, 2)],
        ["Jumlah Frame", total_f],
        ["Ukuran MP4 (MB)", round(size_mp4, 2)],
        ["Ukuran RAW AVI (MB)", round(size_raw, 2)],
        ["Rasio MP4 → RAW", f"{round(ratio, 2)}x"]
    ], columns=["Parameter", "Value"])
    return info_df, total_f

def convert_to_raw(input_file, output_file="input_raw.avi"):
    subprocess.run(["ffmpeg", "-i", input_file, "-c:v", "rawvideo", "-pix_fmt", "yuv420p", "-y", output_file], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    return output_file

def compress_video(input_file):
    bitrates = ["500k", "100k", "50k"]
    for b in bitrates:
        subprocess.run(f"ffmpeg -i {input_file} -b:v {b} -bufsize {b} -y output_{b}.mp4 -loglevel quiet", shell=True)

def extract_frames():
    folders = ["frames/ori", "frames/50k", "frames/100k", "frames/500k"]
    for f in folders:
        if os.path.exists(f): shutil.rmtree(f)
        os.makedirs(f, exist_ok=True)
    subprocess.run('ffmpeg -i input.mp4 -vf "fps=30" frames/ori/frame_%03d.png -y -loglevel quiet', shell=True)
    for b in ["50k", "100k", "500k"]:
        subprocess.run(f'ffmpeg -i output_{b}.mp4 -vf "fps=30" frames/{b}/frame_%03d.png -y -loglevel quiet', shell=True)

# =====================================================
# METRICS LOGIC
# =====================================================
def hitung_metrik(ori, comp):
    if ori.shape != comp.shape: comp = cv2.resize(comp, (ori.shape[1], ori.shape[0]))
    o_g, c_g = cv2.cvtColor(ori, cv2.COLOR_BGR2GRAY), cv2.cvtColor(comp, cv2.COLOR_BGR2GRAY)
    mse = np.mean((o_g.astype(float) - c_g.astype(float))**2)
    psnr = 10 * np.log10(255**2 / mse) if mse > 0 else 100
    blocking = (np.mean(np.abs(np.diff(c_g[:, ::8].astype(float)))) + np.mean(np.abs(np.diff(c_g[::8, :].astype(float))))) / 2
    ringing = np.mean(np.abs(cv2.Canny(c_g, 50, 150).astype(float) - cv2.Canny(o_g, 50, 150).astype(float)))
    cb = np.mean(cv2.absdiff(ori, comp))
    return psnr, mse, blocking, ringing, cb

def calculate_smearing(comp_prev, comp_curr):
    if comp_prev is None or comp_curr is None: return 0
    diff = cv2.absdiff(cv2.cvtColor(comp_curr, cv2.COLOR_BGR2GRAY), cv2.cvtColor(comp_prev, cv2.COLOR_BGR2GRAY))
    return np.mean(diff) / 255.0

def build_cache():
    global cache_metrik, cache_motion, total_frames_global
    cache_metrik, cache_motion = {}, {}
    ori_files = sorted(os.listdir("frames/ori"))
    total_frames_global = len(ori_files)
    for b_label, b_code in [("500 kbps", "500k"), ("100 kbps", "100k"), ("50 kbps", "50k")]:
        prev_frame = None
        for i in range(total_frames_global):
            o = cv2.imread(f"frames/ori/frame_{i+1:03d}.png")
            c = cv2.imread(f"frames/{b_code}/frame_{i+1:03d}.png")
            if o is not None and c is not None:
                cache_metrik[(b_label, i)] = hitung_metrik(o, c)
                cache_motion[(b_label, i)] = calculate_smearing(prev_frame, c)
                prev_frame = c

# =====================================================
# GRADIO ACTIONS
# =====================================================
def process_master(video_file):
    if not video_file: return None, None, None, None, None, None, gr.update(maximum=100)
    shutil.copy(video_file, "input.mp4")
    raw = convert_to_raw("input.mp4")
    compress_video("input.mp4")
    extract_frames()
    build_cache()
    info_df, f_count = get_video_info("input.mp4", raw)
    return info_df, "input.mp4", "output_500k.mp4", "output_100k.mp4", "output_50k.mp4", gr.update(maximum=f_count, value=1)

def analyze_frame(bitrate, f_num, art_type):
    plt.close('all')
    b_code = bitrate.split()[0] + "k"
    ori_bgr = cv2.imread(f"frames/ori/frame_{int(f_num):03d}.png")
    comp_bgr = cv2.imread(f"frames/{b_code}/frame_{int(f_num):03d}.png")

    if ori_bgr is None or comp_bgr is None:
        return None, None, None, None, None, None, None, None

    comp_res = cv2.resize(comp_bgr, (ori_bgr.shape[1], ori_bgr.shape[0]))
    highlight = cv2.cvtColor(comp_res, cv2.COLOR_BGR2RGB)
    c_gray = cv2.cvtColor(comp_res, cv2.COLOR_BGR2GRAY)

    if art_type in ["Blocking", "Semua Artefak"]:
        overlay = highlight.copy()
        for i in range(0, c_gray.shape[0], 8):
            overlay[i, :] = [255, 0, 0]
        for j in range(0, c_gray.shape[1], 8):
            overlay[:, j] = [255, 0, 0]
        highlight = cv2.addWeighted(overlay, 0.5, highlight, 0.5, 0)

    if art_type in ["Ringing", "Semua Artefak"]:
        edges   = cv2.Canny(c_gray, 50, 150)
        edges_d = cv2.dilate(edges, np.ones((5, 5), np.uint8), iterations=2)
        highlight[edges_d == 255] = [0, 255, 0]

    if art_type in ["Color Bleeding", "Semua Artefak"]:
        diff      = cv2.absdiff(ori_bgr, comp_res)
        diff_gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
        _, mask   = cv2.threshold(diff_gray, 25, 255, cv2.THRESH_BINARY)
        mask      = cv2.dilate(mask, np.ones((3, 3), np.uint8), iterations=1)
        highlight[mask == 255] = [255, 220, 0]

    if art_type in ["Motion Smearing", "Semua Artefak"]:
        blur_map = cv2.Laplacian(c_gray, cv2.CV_64F)
        _, mask  = cv2.threshold(np.abs(blur_map).astype(np.uint8), 10, 255, cv2.THRESH_BINARY_INV)
        highlight[mask == 255] = [0, 180, 255]

    m = cache_metrik.get((bitrate, int(f_num)-1), (0,0,0,0,0))
    df = pd.DataFrame({"Metric": ["PSNR", "MSE", "Blocking", "Ringing", "Color Bleeding"], "Value": [round(x, 4) for x in m]})
    figs = []
    frames = np.arange(total_frames_global)

    # --- LOGIKA WARNA GRAFIK BARU ---
    # Hijau (PSNR), Ungu (MSE), Merah (Blocking), Biru (Ringing), Oranye (Color Bleeding)
    warna_grafik = ['#2ecc71', '#9b59b6', '#e74c3c', '#3498db', '#f39c12']

    for i in range(5):
        fig, ax = plt.subplots(figsize=(4, 2))
        data = [cache_metrik.get((bitrate, f), [0]*5)[i] for f in frames]
        ax.plot(frames, data, color=warna_grafik[i], linewidth=1.5)
        # Ubah warna garis penanda frame jadi hitam agar kontras dengan grafik berwarna
        ax.axvline(f_num-1, color='black', linestyle='--', linewidth=1.5)
        ax.set_title(df["Metric"][i], fontsize=10)
        ax.grid(True, alpha=0.3)
        figs.append(fig)

    return cv2.cvtColor(ori_bgr, cv2.COLOR_BGR2RGB), highlight, df, *figs

def render_dash(metric_name):
    plt.close('all')
    fig, ax = plt.subplots(figsize=(10, 4))
    m_map = {"PSNR":0, "MSE":1, "Blocking":2, "Ringing":3, "Color Bleeding":4, "Motion Smearing": 5}
    m_idx = m_map.get(metric_name)
    frames = np.arange(total_frames_global)
    for b in ["500 kbps", "100 kbps", "50 kbps"]:
        data = [cache_motion.get((b, f), 0) for f in frames] if m_idx == 5 else [cache_metrik.get((b, f), [0]*5)[m_idx] for f in frames]
        ax.plot(frames, data, label=b)
    ax.set_title(f"{metric_name} Comparison Across Bitrates"); ax.legend(); ax.grid(True)
    return fig

with gr.Blocks() as demo:
    gr.Markdown("# ━━━━ VIDEO COMPRESSION ANALYSIS ━━━━")

    with gr.Tab("Tab 1: Metadata & Raw"):
        in_vid = gr.File(label="Upload MP4")
        btn_proc = gr.Button("Process Video")
        out_table = gr.Dataframe(label="Metadata Comparison")
        v_orig = gr.Video(label="Video Original")

    with gr.Tab("Tab 2: Compression Comparison"):
        with gr.Row():
            v2_orig = gr.Video(label="Original")
            v2_500 = gr.Video(label="500 kbps")
        with gr.Row():
            v2_100 = gr.Video(label="100 kbps")
            v2_50 = gr.Video(label="50 kbps")

    with gr.Tab("Tab 3: Artifact Explorer"):
        with gr.Row():
            dd_bit = gr.Dropdown(["500 kbps", "100 kbps", "50 kbps"], value="500 kbps", label="Bitrate")
            dd_art = gr.Dropdown(["Blocking", "Ringing", "Color Bleeding", "Motion Smearing", "Semua Artefak"], value="Blocking", label="Artifact Type")
            sld_f = gr.Slider(1, 100, step=1, label="Frame")
        with gr.Row():
            img_o = gr.Image(label="Original Frame"); img_a = gr.Image(label="Artifact Highlight")
        met_df = gr.Dataframe(); btn_art = gr.Button("Analyze Frame")
        with gr.Row():
            p_psnr = gr.Plot(); p_mse = gr.Plot(); p_blk = gr.Plot(); p_rng = gr.Plot(); p_cb = gr.Plot()

    with gr.Tab("Tab 4: Metrics Dashboard"):
        dash_type = gr.Dropdown(["PSNR", "MSE", "Blocking", "Ringing", "Color Bleeding", "Motion Smearing"], label="Metric")
        btn_dash = gr.Button("Show Dashboard"); p_dash = gr.Plot()

    # Events
    btn_proc.click(process_master, [in_vid], [out_table, v_orig, v2_500, v2_100, v2_50, sld_f])
    btn_proc.click(lambda: "input.mp4", None, v2_orig)
    btn_art.click(analyze_frame, [dd_bit, sld_f, dd_art], [img_o, img_a, met_df, p_psnr, p_mse, p_blk, p_rng, p_cb])
    btn_dash.click(render_dash, [dash_type], p_dash)

if __name__ == "__main__":
    demo.launch(share=True, debug=True)

Overwriting gradio_app.py


In [4]:
!python gradio_app.py

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://172cb87ac2bef6004b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Keyboard interruption in main thread... closing server.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 3043, in block_thread
    time.sleep(0.1)
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/gradio_app.py", line 220, in <module>
    demo.launch(share=True, debug=True)
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2950, in launch
    self.block_thread()
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 3047, in block_thread
    self.server.close()
  File "/usr/local/lib/pyth